<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M02/M02_Lab2_Bitcoin_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">📈 M02 Lab 2 — Bitcoin Analyzer: AI + Real Market Data</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Beginner &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Fetch <b>real-time cryptocurrency data</b> from CoinGecko's free API</li>
    <li>Transform raw JSON into a <b>pandas DataFrame</b> and create professional charts</li>
    <li>Send data summaries to GPT and receive <b>structured financial analysis</b></li>
    <li>Use AI to <b>generate visualization code</b> — vibe coding in action</li>
    <li>Build a <b>reusable crypto analyzer function</b> with technical indicators</li>
  </ol>
</div>

In this lab, you will fetch real Bitcoin price data from a free API, send it to GPT for analysis, and generate visual reports — all in Python. By the end, you'll have a reusable tool that rivals paid subscription services.

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Setup</h2>
</div>

Let's install the required packages and configure our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai requests pandas matplotlib
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from datetime import datetime
from openai import OpenAI
from dads5250 import setup_openai, show_response, pp, compare_responses, estimate_cost, show_info, quiz

# This reads your OPENAI_API_KEY from Colab Secrets automatically
client = setup_openai()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Fetching Real Bitcoin Data</h2>
</div>

We'll use [CoinGecko's free API](https://www.coingecko.com/en/api) — no API key needed. It provides historical price and volume data for thousands of cryptocurrencies.

In [ ]:
# ============================================================
# 🌐 Fetch 90 Days of Bitcoin Data from CoinGecko
# ============================================================

url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"
params = {"vs_currency": "usd", "days": 90}

response = requests.get(url, params=params)
data = response.json()

# Parse into a clean DataFrame
df = pd.DataFrame({
    "date": [datetime.fromtimestamp(p[0] / 1000) for p in data["prices"]],
    "price": [p[1] for p in data["prices"]],
    "volume": [v[1] for v in data["total_volumes"][:len(data["prices"])]]
})

print(f"✅ Fetched {len(df)} data points")
print(f"📅 Date range: {df['date'].min().strftime('%Y-%m-%d')} → {df['date'].max().strftime('%Y-%m-%d')}")
print(f"💰 Current price: ${df['price'].iloc[-1]:,.2f}")
print()
df.head(10)

In [ ]:
# ============================================================
# 📊 Basic Price Chart — Professional Dark Theme
# ============================================================

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(df['date'], df['price'], color='#2196F3', linewidth=1.8, alpha=0.95)
ax.fill_between(df['date'], df['price'], alpha=0.08, color='#2196F3')

ax.set_title('Bitcoin Price — Last 90 Days', fontsize=18, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Price (USD)', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(True, alpha=0.15, linestyle='--')
ax.tick_params(labelsize=11)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> CoinGecko's free API has rate limits (~10-30 calls/min). If you get an error, wait 30 seconds and try again. The data updates roughly every few minutes, not in real-time.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ AI Analyzes the Data</h2>
</div>

Now the powerful part: we'll summarize the data and ask GPT to analyze it like a financial analyst. We send a **summary** (not raw data) to keep token costs low.

In [ ]:
# ============================================================
# 📋 Create Data Summary for GPT
# ============================================================

current_price = df['price'].iloc[-1]
max_price = df['price'].max()
min_price = df['price'].min()
avg_price = df['price'].mean()
avg_volume = df['volume'].mean()

# 30-day change
price_30d_ago = df[df['date'] >= df['date'].max() - pd.Timedelta(days=30)]['price'].iloc[0]
change_30d = ((current_price - price_30d_ago) / price_30d_ago) * 100

print("📊 Bitcoin Data Summary")
print(f"   Current price:     ${current_price:,.2f}")
print(f"   90-day high:       ${max_price:,.2f}")
print(f"   90-day low:        ${min_price:,.2f}")
print(f"   Average price:     ${avg_price:,.2f}")
print(f"   30-day change:     {change_30d:+.1f}%")
print(f"   Avg daily volume:  ${avg_volume:,.0f}")

In [ ]:
# ============================================================
# 🤖 Send to GPT for Financial Analysis
# ============================================================

analysis_prompt = f"""You are a financial data analyst. Analyze this Bitcoin price data and provide:
1. Overall trend (bullish/bearish/sideways)
2. Key support and resistance levels
3. 30-day price change and what it means
4. A brief market sentiment summary (2-3 sentences)

Data summary:
- Period: Last 90 days
- Current price: ${current_price:,.2f}
- 90-day high: ${max_price:,.2f}
- 90-day low: ${min_price:,.2f}
- 30-day change: {change_30d:+.1f}%
- Average daily volume: ${avg_volume:,.0f}

Provide your analysis in a structured format."""

ai_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are an expert financial analyst specializing in cryptocurrency markets."},
        {"role": "user", "content": analysis_prompt}
    ],
    max_tokens=500
)

analysis = ai_response.choices[0].message.content
print("🤖 GPT Market Analysis")
print("=" * 60)
print(analysis)
print(f"\n📊 Tokens used: {ai_response.usage.total_tokens}")

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> The AI didn't look at a chart. It analyzed <i>numbers</i>. But its analysis reads like a professional market report. This is the power of combining real data with LLM reasoning — you provide the facts, the AI provides the narrative.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Generating Visual Reports with AI</h2>
</div>

Here's where it gets interesting: we'll ask GPT to **write matplotlib code** for a professional chart, then execute it. This is **vibe coding** — describe what you want, AI writes the code, you run it.

In [ ]:
# ============================================================
# 🎨 Ask GPT to Generate Visualization Code
# ============================================================

viz_prompt = f"""Write Python matplotlib code to create a professional Bitcoin price chart with:
1. Price line in blue (#2196F3) on the main axis
2. 7-day and 30-day moving averages as dashed lines (orange #FF9800 and red #F44336)
3. Volume bars on a secondary axis (semi-transparent green #4CAF50, alpha=0.3)
4. A shaded region between the moving averages
5. Clean dark theme (plt.style.use('dark_background')), proper labels, legend
6. Title: 'Bitcoin Price Analysis — Last 90 Days'
7. Figure size: (14, 7)
8. Subtle grid lines (alpha=0.15)
9. Format y-axis as USD with commas
10. Use fig.autofmt_xdate() for clean date labels

The data is in a pandas DataFrame called 'df' with columns: 'date', 'price', 'volume'.
Use matplotlib.ticker as mticker (already imported).
End with plt.tight_layout() and plt.show().
Return ONLY the Python code, no explanation, no markdown fences."""

viz_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": viz_prompt}],
    max_tokens=800
)

chart_code = viz_response.choices[0].message.content.strip()

# Remove markdown fences if present
if chart_code.startswith("```"):
    chart_code = "\n".join(chart_code.split("\n")[1:])
if chart_code.endswith("```"):
    chart_code = chart_code[:-3].strip()

print("📝 AI-Generated Code:")
print("=" * 60)
print(chart_code)

In [ ]:
# ============================================================
# ▶️ Execute the AI-Generated Chart Code
# ============================================================

exec(chart_code)

<div style="background: #fff8e1; border-left: 4px solid #f9a825; padding: 12px 16px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <b>💡 Pro Tip:</b> We asked the AI to <i>write the visualization code</i>, not just describe it. This is <b>vibe coding</b> — describe what you want, AI writes the code, you run it. If the chart doesn't look right, just refine your prompt and regenerate.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ Building a Reusable Analyzer</h2>
</div>

Let's wrap everything into a clean, reusable function. One call fetches data, runs AI analysis, and generates a professional chart.

In [ ]:
# ============================================================
# 🔧 Reusable Crypto Analyzer Function
# ============================================================

def crypto_analyzer(coin="bitcoin", days=90):
    """Fetch real crypto data, run AI analysis, generate charts."""

    # 1. Fetch data from CoinGecko
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart"
    params = {"vs_currency": "usd", "days": days}
    resp = requests.get(url, params=params)
    data = resp.json()

    coin_df = pd.DataFrame({
        "date": [datetime.fromtimestamp(p[0] / 1000) for p in data["prices"]],
        "price": [p[1] for p in data["prices"]],
        "volume": [v[1] for v in data["total_volumes"][:len(data["prices"])]]
    })

    # 2. Create summary statistics
    cur = coin_df['price'].iloc[-1]
    hi = coin_df['price'].max()
    lo = coin_df['price'].min()
    avg_vol = coin_df['volume'].mean()
    p30 = coin_df[coin_df['date'] >= coin_df['date'].max() - pd.Timedelta(days=30)]['price'].iloc[0]
    chg = ((cur - p30) / p30) * 100

    summary = (
        f"Coin: {coin.title()} | Period: Last {days} days\n"
        f"Current price: ${cur:,.2f} | High: ${hi:,.2f} | Low: ${lo:,.2f}\n"
        f"30-day change: {chg:+.1f}% | Avg daily volume: ${avg_vol:,.0f}"
    )

    # 3. Get AI analysis
    ai_resp = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": "You are an expert crypto analyst. Be concise."},
            {"role": "user", "content": f"Analyze this data and give: trend, support/resistance, sentiment (3 sentences max).\n\n{summary}"}
        ],
        max_tokens=300
    )
    analysis_text = ai_resp.choices[0].message.content

    # 4. Generate professional chart
    plt.style.use('dark_background')
    fig, ax1 = plt.subplots(figsize=(14, 7))

    ax1.plot(coin_df['date'], coin_df['price'], color='#2196F3', linewidth=1.8, label='Price')

    # Moving averages
    if len(coin_df) > 7:
        ma7 = coin_df['price'].rolling(window=min(7, len(coin_df))).mean()
        ax1.plot(coin_df['date'], ma7, color='#FF9800', linewidth=1.2, linestyle='--', label='7-day MA', alpha=0.8)
    if len(coin_df) > 30:
        ma30 = coin_df['price'].rolling(window=min(30, len(coin_df))).mean()
        ax1.plot(coin_df['date'], ma30, color='#F44336', linewidth=1.2, linestyle='--', label='30-day MA', alpha=0.8)

    ax1.set_title(f'{coin.title()} Price Analysis — Last {days} Days', fontsize=18, fontweight='bold', pad=15)
    ax1.set_xlabel('Date', fontsize=13)
    ax1.set_ylabel('Price (USD)', fontsize=13)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax1.grid(True, alpha=0.15, linestyle='--')
    ax1.legend(loc='upper left', fontsize=11)

    # Volume on secondary axis
    ax2 = ax1.twinx()
    ax2.bar(coin_df['date'], coin_df['volume'], alpha=0.15, color='#4CAF50', width=0.8)
    ax2.set_ylabel('Volume (USD)', fontsize=13, alpha=0.5)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e9:.1f}B'))
    ax2.tick_params(axis='y', labelsize=10, colors='#4CAF50')

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    # 5. Return everything
    print(f"\n🤖 AI Analysis for {coin.title()}")
    print("=" * 60)
    print(analysis_text)

    return {"df": coin_df, "summary": summary, "analysis": analysis_text}

In [ ]:
# ============================================================
# 🪙 Demo: Analyze Bitcoin (90 days)
# ============================================================

btc_result = crypto_analyzer("bitcoin", 90)

In [ ]:
# ============================================================
# 💎 Demo: Analyze Ethereum (30 days)
# ============================================================

eth_result = crypto_analyzer("ethereum", 30)

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">6️⃣ Adding Technical Indicators</h2>
</div>

Let's ask AI to calculate and explain key technical indicators: **Moving Averages**, **RSI** (Relative Strength Index), and **Bollinger Bands**. Then we'll plot them all on one professional chart.

You just built a tool that would cost hundreds of dollars as a subscription service.

In [ ]:
# ============================================================
# 📐 Calculate Technical Indicators
# ============================================================

# Moving Averages
df['ma7'] = df['price'].rolling(window=7).mean()
df['ma30'] = df['price'].rolling(window=30).mean()

# RSI (Relative Strength Index) — 14-period
delta = df['price'].diff()
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['rsi'] = 100 - (100 / (1 + rs))

# Bollinger Bands — 20-period, 2 standard deviations
df['bb_mid'] = df['price'].rolling(window=20).mean()
bb_std = df['price'].rolling(window=20).std()
df['bb_upper'] = df['bb_mid'] + 2 * bb_std
df['bb_lower'] = df['bb_mid'] - 2 * bb_std

print("✅ Technical indicators calculated:")
print(f"   RSI (latest): {df['rsi'].iloc[-1]:.1f}")
print(f"   7-day MA:     ${df['ma7'].iloc[-1]:,.2f}")
print(f"   30-day MA:    ${df['ma30'].iloc[-1]:,.2f}")
print(f"   Bollinger:    ${df['bb_lower'].iloc[-1]:,.2f} — ${df['bb_upper'].iloc[-1]:,.2f}")

In [ ]:
# ============================================================
# 📊 Professional Technical Analysis Chart
# ============================================================

plt.style.use('dark_background')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# --- Top: Price + MAs + Bollinger Bands ---
ax1.plot(df['date'], df['price'], color='#2196F3', linewidth=1.8, label='Price', zorder=3)
ax1.plot(df['date'], df['ma7'], color='#FF9800', linewidth=1.2, linestyle='--', label='7-day MA', alpha=0.85)
ax1.plot(df['date'], df['ma30'], color='#F44336', linewidth=1.2, linestyle='--', label='30-day MA', alpha=0.85)

# Bollinger Bands
ax1.plot(df['date'], df['bb_upper'], color='#9E9E9E', linewidth=0.8, alpha=0.5)
ax1.plot(df['date'], df['bb_lower'], color='#9E9E9E', linewidth=0.8, alpha=0.5)
ax1.fill_between(df['date'], df['bb_upper'], df['bb_lower'], alpha=0.06, color='#9E9E9E', label='Bollinger Bands')

ax1.set_title('Bitcoin Technical Analysis — Last 90 Days', fontsize=18, fontweight='bold', pad=15)
ax1.set_ylabel('Price (USD)', fontsize=13)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(loc='upper left', fontsize=10, framealpha=0.3)
ax1.grid(True, alpha=0.15, linestyle='--')

# --- Bottom: RSI ---
ax2.plot(df['date'], df['rsi'], color='#AB47BC', linewidth=1.5, label='RSI (14)')
ax2.axhline(y=70, color='#F44336', linestyle='--', alpha=0.5, linewidth=0.8)
ax2.axhline(y=30, color='#4CAF50', linestyle='--', alpha=0.5, linewidth=0.8)
ax2.fill_between(df['date'], df['rsi'], 70, where=(df['rsi'] >= 70), alpha=0.2, color='#F44336')
ax2.fill_between(df['date'], df['rsi'], 30, where=(df['rsi'] <= 30), alpha=0.2, color='#4CAF50')
ax2.set_ylabel('RSI', fontsize=13)
ax2.set_xlabel('Date', fontsize=13)
ax2.set_ylim(10, 90)
ax2.legend(loc='upper left', fontsize=10, framealpha=0.3)
ax2.grid(True, alpha=0.15, linestyle='--')

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 🤖 Ask AI to Explain the Technical Indicators
# ============================================================

tech_prompt = f"""You are a financial educator. Explain these Bitcoin technical indicators to a graduate student in 2-3 sentences each:

1. RSI is currently {df['rsi'].iloc[-1]:.1f} — what does this mean?
2. The 7-day MA (${df['ma7'].iloc[-1]:,.2f}) vs 30-day MA (${df['ma30'].iloc[-1]:,.2f}) — bullish or bearish cross?
3. Current price ${current_price:,.2f} relative to Bollinger Bands (${df['bb_lower'].iloc[-1]:,.2f} — ${df['bb_upper'].iloc[-1]:,.2f}) — what does the position tell us?

Keep it practical and jargon-free."""

tech_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": tech_prompt}],
    max_tokens=400
)

print("🤖 AI Explains the Indicators")
print("=" * 60)
print(tech_response.choices[0].message.content)

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "Why do we send a data SUMMARY to the API instead of raw price data?",
        "options": [
            "Raw data is more accurate for the model",
            "Token limits and cost — 90 days of hourly data would be thousands of tokens",
            "The API only accepts summary statistics",
            "Raw data causes hallucinations"
        ],
        "answer": 1,
        "explanation": "Sending 90 days of hourly price data would consume thousands of tokens (= higher cost) and likely exceed context limits. A concise summary gives the model all the key information it needs at a fraction of the cost."
    },
    {
        "q": "What is a moving average and why is it useful for trend analysis?",
        "options": [
            "It predicts future prices with 95% accuracy",
            "It smooths out short-term noise to reveal the overall price direction",
            "It calculates the median price over a fixed window",
            "It measures trading volume relative to price"
        ],
        "answer": 1,
        "explanation": "A moving average averages the price over a rolling window (e.g., 7 days). This smooths out daily fluctuations and reveals the underlying trend — making it easier to see if prices are generally going up, down, or sideways."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Multi-Coin Comparison

Extend the analyzer to compare two cryptocurrencies. Replace each `-----` with the correct value.

**Your task:**
1. Fetch data for a second coin (e.g., Ethereum or Solana)
2. Ask AI to compare both coins and recommend which shows stronger momentum
3. Generate a comparison chart with both coins normalized to percentage change

In [ ]:
# ============================================================
# 🔧 Exercise: Fetch a Second Coin
# ============================================================
# Replace each ----- with the correct value

# Step 1: Fetch Ethereum data (use the same CoinGecko API pattern)
eth_url = "https://api.coingecko.com/api/v3/coins/-----/market_chart"  # Which coin ID?
eth_params = {"vs_currency": "usd", "days": -----}                     # How many days?

eth_resp = requests.get(eth_url, params=eth_params)
eth_data = eth_resp.json()

eth_df = pd.DataFrame({
    "date": [datetime.fromtimestamp(p[0] / 1000) for p in eth_data["prices"]],
    "price": [p[1] for p in eth_data["prices"]],
})

print(f"✅ Fetched {len(eth_df)} Ethereum data points")
print(f"💰 Current ETH price: ${eth_df['price'].iloc[-1]:,.2f}")

In [ ]:
# ============================================================
# 🔧 Exercise: Ask AI to Compare Both Coins
# ============================================================

# Step 2: Create summaries for both coins
btc_summary = f"Bitcoin: ${df['price'].iloc[-1]:,.2f}, 30d change: {change_30d:+.1f}%"

eth_price_30d_ago = eth_df[eth_df['date'] >= eth_df['date'].max() - pd.Timedelta(days=30)]['price'].iloc[0]
eth_change_30d = ((eth_df['price'].iloc[-1] - eth_price_30d_ago) / eth_price_30d_ago) * 100
eth_summary = f"Ethereum: ${eth_df['price'].iloc[-1]:,.2f}, 30d change: {eth_change_30d:+.1f}%"

compare_prompt = f"""Compare these two cryptocurrencies and recommend which shows stronger momentum.
Keep your response to 4-5 sentences.

{btc_summary}
{eth_summary}"""

compare_response = client.chat.completions.create(
    model="-----",                                          # Which model?
    messages=[{"role": "-----", "content": compare_prompt}],  # What role?
    max_tokens=300
)

print("🤖 AI Comparison")
print("=" * 60)
print(compare_response.-----[0].message.-----)               # Access the content

In [ ]:
# ============================================================
# 🔧 Exercise: Normalized Comparison Chart
# ============================================================

# Step 3: Normalize both to percentage change from day 1
btc_norm = (df['price'] / df['price'].iloc[0] - 1) * 100
eth_norm = (eth_df['price'] / eth_df['price'].iloc[0] - 1) * 100

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(df['date'], btc_norm, color='-----', linewidth=1.8, label='Bitcoin')       # What color for BTC?
ax.plot(eth_df['date'], eth_norm, color='-----', linewidth=1.8, label='Ethereum')  # What color for ETH?
ax.axhline(y=0, color='white', linewidth=0.5, alpha=0.3)

ax.set_title('Bitcoin vs Ethereum — Normalized % Change (90 Days)', fontsize=18, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Change (%)', fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.15, linestyle='--')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

**Expected Output:** A chart showing both coins' performance normalized to percentage change from the start of the 90-day period. This makes it easy to compare which coin performed better, regardless of their absolute price difference ($80K+ vs $2K+).

**📝 Your Observations:** *(double-click to edit this cell)*

1. Which coin showed stronger momentum over 90 days? Did the AI agree with what the chart shows? _[Your answer]_

2. Why is normalizing to percentage change important when comparing assets with very different prices? _[Your answer]_

3. What other data would make the AI's comparison more useful? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activities</h3>
  <ol style="font-size: 14px;">
    <li>Try different time periods (7 days vs 365 days) — how does the AI's analysis change? Does a shorter window produce more actionable insights?</li>
    <li>Ask the AI to generate a <b>1-paragraph investment memo</b> suitable for a portfolio manager. How does framing the audience change the output?</li>
    <li>Try analyzing a less popular coin (e.g., <code>"dogecoin"</code>, <code>"cardano"</code>) — does the AI still produce useful analysis with less mainstream assets?</li>
    <li>Modify the <code>crypto_analyzer</code> function to return a risk score (1-10) based on RSI and volatility.</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You built a professional crypto analyzer — real data, AI analysis, and generated visualizations.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>Real data + AI analysis + code generation</b> = professional analytical tool</li>
      <li>Send <b>summaries</b> to the API, not raw data — saves tokens and cost</li>
      <li><b>Vibe coding:</b> describe the chart you want, AI writes the matplotlib code</li>
      <li>The same pattern works for <b>any data domain</b> — stocks, weather, sports, health</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M03 — Prompt Engineering, Structured Output &amp; Function Calling</p>
</div>